In [12]:
# Downgrade kilosort to the version compatible with SpikeInterface 0.103.0
# Run this once, then restart the kernel before running the rest of the notebook
! pip install kilosort==4.0.34

   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   - -------------------------------------- 0.8/15.8 MB 4.8 MB/s eta 0:00:04
   ----- ---------------------------------- 2.1/15.8 MB 5.1 MB/s eta 0:00:03
   ------- -------------------------------- 3.1/15.8 MB 5.4 MB/s eta 0:00:03
   ----------- ---------------------------- 4.5/15.8 MB 5.6 MB/s eta 0:00:03
   -------------- ------------------------- 5.8/15.8 MB 5.9 MB/s eta 0:00:02
   ----------------- ---------------------- 7.1/15.8 MB 5.8 MB/s eta 0:00:02
   -------------------- ------------------- 8.1/15.8 MB 5.7 MB/s eta 0:00:02
   ----------------------- ---------------- 9.4/15.8 MB 5.8 MB/s eta 0:00:02
   --------------------------- ------------ 10.7/15.8 MB 5.9 MB/s eta 0:00:01
   ------------------------------ --------- 12.1/15.8 MB 5.9 MB/s eta 0:00:01
   --------------------------------- ------ 13.4/15.8 MB 5.9 MB/s eta 0:00:01
   ------------------------------------- -- 14.7/15.8 MB 6.0 MB/s eta 0:00:01
  

In [2]:
import spikeinterface.widgets as sw
print(sw.__version__)


AttributeError: module 'spikeinterface.widgets' has no attribute '__version__'

In [1]:
import spikeinterface as si
import kilosort

print(si.__version__)
print(kilosort.__version__)


C:\Users\labuser\miniforge3\envs\si_pipeline\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0.103.0
4.0.34


In [2]:
import spikeinterface.full as si
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [3]:
from pathlib import Path
import shutil

spikeglx_folder = Path(r'C:\Users\labuser\Ilaria\Project\small_rec\2142_g0\2142_g0_imec0')
base_folder     = Path(r'C:\Users\labuser\Ilaria\Project\processed\2142_output')

# Check 1: raw data folder exists?
print(f"SpikeGLX folder exists : {spikeglx_folder.exists()}")

# Check 2: list what's inside
print("\nFiles found:")
for f in sorted(spikeglx_folder.iterdir()):
    size_gb = f.stat().st_size / 1e9
    print(f"  {f.name:50s} {size_gb:.2f} GB")

# Check 3: .bin and .meta both there?
bin_files  = list(spikeglx_folder.glob('*.ap.bin'))
meta_files = list(spikeglx_folder.glob('*.ap.meta'))
print(f"\n.ap.bin  found: {len(bin_files)}")
print(f".ap.meta found: {len(meta_files)}")

# Check 4: create output folder
base_folder.mkdir(parents=True, exist_ok=True)
print(f"\nOutput folder created : {base_folder.exists()}")

# Check 5: disk space
total, used, free = shutil.disk_usage('C:\\')
print(f"Free space on C:      : {free/1e9:.0f} GB")
print(f"\n✓ Ready — safe to run the pipeline"
      if spikeglx_folder.exists()
         and len(bin_files) > 0
         and len(meta_files) > 0
      else "\n✗ Something is wrong — fix above before continuing")

SpikeGLX folder exists : True

Files found:
  2142_g0_Kilosort                                   0.00 GB
  2142_g0_t0.imec0.ap.bin                            0.09 GB
  2142_g0_t0.imec0.ap.meta                           0.00 GB
  2142_g0_t0.imec0.lf.bin                            0.01 GB
  2142_g0_t0.imec0.lf.meta                           0.00 GB
  numpy_data.h5                                      0.01 GB

.ap.bin  found: 1
.ap.meta found: 1

Output folder created : True
Free space on C:      : 892 GB

✓ Ready — safe to run the pipeline


In [4]:
import spikeinterface.full as si
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

#YOUR PATHS 
spikeglx_folder = Path(r'C:\Users\labuser\Ilaria\Project\small_rec\2142_g0\2142_g0_imec0')
base_folder     = Path(r'C:\Users\labuser\Ilaria\Project\processed\2142_output')

# check streams
stream_names, stream_ids = si.get_neo_streams(
    'spikeglx', spikeglx_folder)
print(stream_names)

# load
raw_rec = si.read_spikeglx(spikeglx_folder,
    stream_name='imec0.ap', load_sync_channel=False)
print(raw_rec)

# preprocessing 
rec1 = si.highpass_filter(raw_rec,
           freq_min=300.)             # ← CHANGE 2: 400→300 Hz
                                       
bad_channel_ids, _ = si.detect_bad_channels(rec1)
rec2 = rec1.remove_channels(bad_channel_ids)
#rec3 = si.phase_shift(rec2)
rec4 = si.common_reference(rec2,
           operator="median", reference="global")
rec = rec4

# save binary — keep, but change n_jobs and folder
job_kwargs = dict(
    n_jobs=4,                         # ← CHANGE 3: 40→4 threads
    chunk_duration='1s',
    progress_bar=True)
rec = rec.save(
    folder=base_folder / 'preprocess 6', # ← uses your base_folder
    format='binary',
    **job_kwargs)

# run sorter 
sorting = si.run_sorter(
    'kilosort4', rec,
    folder=base_folder / 'kilosort_output', # ← CHANGE 4
    remove_existing_folder=True, verbose=True)
print(sorting)

['imec0.ap', 'imec0.lf', 'imec0.ap-SYNC', 'imec0.lf-SYNC']
SpikeGLXRecordingExtractor: 384 channels - 30000.386555 Hz - 1 segments - 120,001 samples - 4.00s 
                            int16 dtype - 87.89 MiB


C:\Users\labuser\miniforge3\envs\si_pipeline\lib\site-packages\spikeinterface\extractors\neoextractors\spikeglx.py:99: UserWarning: Unable to find inter-sample shifts in the Neuropixels probe metadata. The sample shifts will not be loaded. 
  sample_shifts = get_neuropixels_sample_shifts_from_probe(probe, stream_name=self.stream_name)


write_binary_recording 
engine=process - n_jobs=4 - samples_per_chunk=30,000 - chunk_memory=21.92 MiB - total_memory=87.66 MiB - chunk_duration=1.00s (999.99 ms)


write_binary_recording (workers: 4 processes): 100%|█████████████████████████████████████| 5/5 [00:00<00:00, 10.61it/s]
kilosort.run_kilosort:  
kilosort.run_kilosort: Computing preprocessing variables.
kilosort.run_kilosort: ----------------------------------------
kilosort.run_kilosort: N samples: 120000
kilosort.run_kilosort: N seconds: 3.9999484599974346
kilosort.run_kilosort: N batches: 2
kilosort.run_kilosort: Preprocessing filters computed in  0.77s; total  0.77s
kilosort.run_kilosort:  
kilosort.run_kilosort: Resource usage after preprocessing
kilosort.run_kilosort: ********************************************************
kilosort.run_kilosort: CPU usage:    12.10 %
kilosort.run_kilosort: Mem used:     38.00 %     |      12.05 GB
kilosort.run_kilosort: Mem avail:    19.67 / 31.71 GB
kilosort.run_kilosort: ------------------------------------------------------
kilosort.run_kilosort: GPU usage:    N/A
kilosort.run_kilosort: GPU memory:   N/A
kilosort.run_kilosort: ***************

kilosort4 run time 104.46s
KiloSortSortingExtractor: 81 units - 1 segments - 30.0kHz


In [4]:
# ── In your Jupyter notebook, after loading ──────────────────
import spikeinterface.widgets as sw
import matplotlib.pyplot as plt

# ── Plot 1: Raw traces — first 2 seconds, first 10 channels ──
# From: SI quickstart tutorial "Let's use the widgets module
#        to visualize the traces"
sw.plot_traces(
    raw_rec,
    time_range=(0, 2),         # first 2 seconds
    channel_ids=raw_rec.channel_ids[:10],  # first 10 ch
    backend='matplotlib'       # static image
)
plt.suptitle('Raw recording — first 10 channels')
plt.tight_layout()

# ── Plot 2: Same but interactive — scroll through channels ───
# Change backend to ipywidgets for interactivity
sw.plot_traces(
    raw_rec,
    backend='ipywidgets'        # interactive scrollable
)
# This gives you a scrollable viewer — drag time axis,
# select channels, zoom in on spikes

# ── Plot 3: Compare raw vs preprocessed side by side ─────────
# From: SI widgets tutorial "rec_dict for multiple layers"
rec_dict = {
    'raw':  raw_rec,
    'preprocessed': rec    # your filtered+CAR recording
}
sw.plot_traces(
    recording=rec_dict,
    time_range=(0, 2),
    channel_ids=raw_rec.channel_ids[:6],
    backend='ipywidgets'
)
# Toggle between raw and preprocessed with the dropdown

# ── Plot 4: Activity map — which channels have signal ─────────
sw.plot_activity_map(
    rec,
    backend='matplotlib'
)
plt.title('Channel activity map')
# Shows a heatmap of signal power per electrode position
# Useful to spot dead channels before bad channel removal


NameError: name 'raw_rec' is not defined

In [5]:
# ── Option B: sigui — interactive desktop app (recommended) ──
# Launch from terminal (with si_pipeline active):

# First export the analyzer so sigui can find it:
# (if not already saved as binary_folder)
analyzer_path = base_folder / 'sorting_analyzer'

# Then open from terminal:
# sigui path/to/sorting_analyzer
#
# OR launch from inside Python:
import spikeinterface_gui
spikeinterface_gui.run(analyzer)

# sigui shows: waveforms · ISI · amplitude · probe map
# Click any unit to inspect it
# No export step needed — reads analyzer directly


NameError: name 'base_folder' is not defined